# AKRM v2 — Stage 2: Learning Protocol Research

**Research Objective:** *We investigate which learning protocol most effectively incorporates uncertainty-guided learning experiences while preserving previously acquired knowledge.*

**Experimental Design:**
- **Independent Variable:** Learning Protocol (5 levels)
- **Controlled Variables:** Same model, dataset (CIFAR-10), seeds, AKRM policy (heuristic), providers
- **Dependent Variables:** Accuracy, F1, ECE, Mean Entropy

> Run all cells **in order**. Each protocol run takes ~15-25 min on T4 GPU.

## Cell 1 — Setup

In [ ]:
!git clone https://github.com/Captdumbledore/Uncertainty-driven-learning-framework_MVP.git
%cd Uncertainty-driven-learning-framework_MVP
!pip install -q -r requirements.txt
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 2 — Protocol 1: Offline Retraining (Control)

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy heuristic \
    --learning_protocol offline \
    --output_dir outputs_protocol_offline

## Cell 3 — Protocol 2: Replay Buffer

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy heuristic \
    --learning_protocol replay \
    --output_dir outputs_protocol_replay

## Cell 4 — Protocol 3: Mixed Replay

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy heuristic \
    --learning_protocol mixed \
    --output_dir outputs_protocol_mixed

## Cell 5 — Protocol 4: Curriculum Replay

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy heuristic \
    --learning_protocol curriculum \
    --output_dir outputs_protocol_curriculum

## Cell 6 — Protocol 5: Knowledge Distillation

In [ ]:
!python main.py \
    --dataset CIFAR10 \
    --akrm_policy heuristic \
    --learning_protocol distillation \
    --output_dir outputs_protocol_distillation

## Cell 7 — Results Comparison

Compare Pipeline C accuracy across all 5 learning protocols.

In [ ]:
import json, os

PROTOCOLS = [
    ('Offline Retraining',     'outputs_protocol_offline'),
    ('Replay Buffer',          'outputs_protocol_replay'),
    ('Mixed Replay',           'outputs_protocol_mixed'),
    ('Curriculum Replay',      'outputs_protocol_curriculum'),
    ('Knowledge Distillation', 'outputs_protocol_distillation'),
]

def load_pipeline_c(output_dir):
    path = os.path.join(output_dir, 'aggregated_results.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        data = json.load(f)
    return data.get('summary', {}).get('C', None)

def load_baseline(output_dir):
    path = os.path.join(output_dir, 'aggregated_results.json')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        data = json.load(f)
    return data.get('summary', {}).get('A', None)

print('\n' + '='*80)
print('  STAGE 2 RESULTS — LEARNING PROTOCOL COMPARISON (Pipeline C only)')
print('='*80)
print(f"  {'Protocol':<28} {'Acc (mean +/- std)':<22} {'F1':<22} {'ECE'}")
print('-'*80)

# Print baseline first
baseline = load_baseline(PROTOCOLS[0][1])
if baseline:
    acc = baseline.get('accuracy', {})
    f1  = baseline.get('f1', {})
    ece = baseline.get('ece', {})
    print(f"  {'[Baseline] No Retrain':<28} "
          f"{acc.get('mean',0):.4f} +/- {acc.get('std',0):.4f}     "
          f"{f1.get('mean',0):.4f} +/- {f1.get('std',0):.4f}     "
          f"{ece.get('mean',0):.4f} +/- {ece.get('std',0):.4f}")
    print('-'*80)
    baseline_acc = acc.get('mean', 0)
else:
    baseline_acc = None

best_acc = -1
best_name = ''

for name, output_dir in PROTOCOLS:
    m = load_pipeline_c(output_dir)
    if m is None:
        print(f"  {name:<28} NOT FOUND")
        continue
    acc = m.get('accuracy', {})
    f1  = m.get('f1', {})
    ece = m.get('ece', {})
    mean_acc = acc.get('mean', 0)
    delta = ''
    if baseline_acc is not None:
        d = mean_acc - baseline_acc
        delta = f'  ({d:+.4f} vs baseline)'
    print(f"  {name:<28} "
          f"{mean_acc:.4f} +/- {acc.get('std',0):.4f}     "
          f"{f1.get('mean',0):.4f} +/- {f1.get('std',0):.4f}     "
          f"{ece.get('mean',0):.4f} +/- {ece.get('std',0):.4f}{delta}")
    if mean_acc > best_acc:
        best_acc = mean_acc
        best_name = name

print('\n' + '='*80)
print(f'  BEST PROTOCOL: {best_name} (Acc = {best_acc:.4f})')
if baseline_acc is not None:
    if best_acc > baseline_acc:
        print(f'  -> CROSSED THE BASELINE ({baseline_acc:.4f})!')
    else:
        print(f'  -> Still below baseline ({baseline_acc:.4f}), gap = {baseline_acc - best_acc:.4f}')
print('='*80)